# einops-repeat — ex8: nearest-neighbor upsample pyramid (8 → 16 → 32) with side-by-side viz

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-repeat`. Running the final beacon cell reports progress against the `Einops: Repeat` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat`** (exercise 8). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat"
DD_SUBTOPIC = "Einops: Repeat"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.repeat — quick refresher

`repeat(tensor, pattern, **axes_lengths)` introduces new axes or stretches existing ones:
1. **New axis** — `'h w -> b h w'` with `b=4` broadcasts across a new leading dim.
2. **Stretch (nearest-neighbor)** — `'h w -> (h r) w'` with `r=2` makes each row appear twice in a contiguous block (rows 0,0,1,1,2,2,...).
3. **Tile** — `'h w -> h (r w)'` with `r=2` concatenates two full copies side-by-side (cols 0..w-1, then 0..w-1 again).

Stretch vs tile: in the composite `(a b)` the axis written **first varies slower**. `(h r)` puts source row 0 at output rows `0..r-1`; `(r h)` puts source row 0 at output rows `0, h, 2h, ...`. The new exercises lean on this distinction repeatedly.

### Exercise 8 — nearest-neighbor upsample pyramid (8 → 16 → 32) with side-by-side viz

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Compose two stretches in a single repeat pattern to perform a 2x nearest-neighbor upsample, then apply it iteratively to build an image pyramid.
> Keywords: upsample, pyramid, stretch, composition, visualization
> ```

**KCs targeted:** `repeat-stretch-vs-tile`, `repeat-axis-composition`, `repeat-iterated-application`

Implement `ex8_nn_upsample_pyramid(img, levels)`.

Input: `img` is a `(H, W)` 2D tensor. Output: a Python list of length `levels + 1`. Element 0 is the original `img`; element `k` is `img` upsampled by `2**k` using nearest-neighbor (each source pixel becomes a `2x2` block of identical values at level 1, a `4x4` block at level 2, etc.).

Build each level by calling `einops.repeat` on the **previous level** with a single pattern that stretches both axes by 2. Do not call `F.interpolate`, `kron`, or write a Python `for`-loop over individual pixels. Loop over levels is fine.

The test cell plots all levels side-by-side using matplotlib `imshow` with `interpolation='nearest'` so you can confirm the staircase scaling is exact, not blurred.

In [ ]:
def ex8_nn_upsample_pyramid(img: Tensor, levels: int) -> list[Tensor]:
    out = [img]
    for _ in range(levels):
        prev = out[-1]
        upsampled = repeat(prev, 'h w -> (h r1) (w r2)', r1=2, r2=2)
        out.append(upsampled)
    return out


<details><summary>Solution</summary>

```python
def ex8_nn_upsample_pyramid(img: Tensor, levels: int) -> list[Tensor]:
    out = [img]
    for _ in range(levels):
        prev = out[-1]
        upsampled = repeat(prev, 'h w -> (h r1) (w r2)', r1=2, r2=2)
        out.append(upsampled)
    return out
```

**Pattern recap.** `'h w -> (h r1) (w r2)'` with `r1=r2=2` is the canonical 2x NN-upsample. The composite `(h r1)` puts source row `i` at output rows `2i, 2i+1` — i.e., **stretch** semantics (each row appears as a contiguous 2-block), not **tile** semantics (rows interleaved). If you instead wrote `'h w -> (r1 h) (r2 w)'`, you'd get an image where each source row appears at output rows `i` and `H+i` — which is **not** nearest-neighbor upsample; it's a checkerboard-replica.

**Why iterative.** Each level depends on the previous, so the function loop is structural (over levels) rather than over pixels.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex8'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex8',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()